# Preparing the training data

### Setting up the data and tools

In [1]:
# Importing the notebook with common setup 
%run 'setup_mc.ipynb'

Welcome to JupyROOT 6.28/00


Invoke (root_datasets, pandas_datasets) = load_data(inclmc_type="23903000") to load datasets
Invoke (root_datasets, pandas_datasets) = load_data(inclmc_type="23903003") to load dataset for double charm
Invoke  df_signal_23903000 = load_signal_from_inclMC() to load signal from 23903000 Inclusive MC
or
Invoke  df_signal = load_signal_all()
Invoke  df_background = load_background_category(category)


### Loading the dataframe with signal and all types of backgrounds

In [2]:
%%time
dfall = load_complete_df()

CPU times: user 13.6 s, sys: 652 ms, total: 14.3 s
Wall time: 28.3 s


### Checking the amount of data per eventIndex

In [3]:
df = dfall#.query('eventIndex == 0')

In [4]:
mygroupby(df, 'signal')

,signal,count,Percentage,cumulative %
0,0,637271,98.072019,98.072019
1,1,12528,1.927981,100.000000


In [5]:
mygroupby(df.query("eventIndex == 0"), 'signal')

,signal,count,Percentage,cumulative %
0,0,318712,98.070361,98.070361
1,1,6271,1.929639,100.000000


In [6]:
mygroupby(df.query("eventIndex == 1"), 'signal')

,signal,count,Percentage,cumulative %
0,0,318559,98.073679,98.073679
1,1,6257,1.926321,100.000000


# Preparing the data with the appropriate columns

In [7]:
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import RocCurveDisplay, accuracy_score
from sklearn.metrics import roc_curve, auc
from sklearn.utils import shuffle

### Columns of interest

In [8]:
train_columns = [
    "Y_0_40_nc_mult",
    "Y_0_20_cc_mult",
    "Y_0_20_cc_PZ",
    "Y_0_30_nc_PZ",
    "Y_0_40_nc_PZ",
    "min_m2pi",
    "max_m2pi",
    "missing_mass_2",
    "B_BPVVDR",
    "B_M",
    "B_correctedMass",
    "log(abs(PBsn))",
    "log(abs(PBv/B_P))",
    "log(abs(PBvn/B_P))",
    "log(abs((PBsn-PBvn)/PBvn))",
    "log(sqrt(abs(mDs2vn)))",
    "mN2v",
    "log(Y_PE)",
    "BDT_Iso",
    "B_pT_Bdir",
    "Y_BPVVDR",
    "missing_pY_mass",
    "Y_correctedMass",
]

### Split the dataset in training and test sets

In [9]:
from sklearn.model_selection import train_test_split
df = shuffle(df)

#tmp, test = train_test_split(df, test_size=0.2)
train = df.query("eventIndex==0")
tmp = df.query("eventIndex==1")

valid, test = train_test_split(tmp, test_size=0.5)

In [10]:
Xtrain = train[train_columns]
ytrain = train["signal"]
Xvalid = valid[train_columns]
yvalid = valid["signal"]

In [11]:
Xtrain.shape

(324983, 23)

In [12]:
Xvalid.shape

(162408, 23)

In [13]:
Xtest = test[train_columns]
ytest = test["signal"]

In [14]:
Xtest.shape

(162408, 23)

### Scaling the features for easier training

In [15]:
%%time
from sklearn import preprocessing
scaler = preprocessing.StandardScaler().fit(Xtrain)
Xtrain_scaled = scaler.transform(Xtrain)
Xvalid_scaled = scaler.transform(Xvalid)
Xtest_scaled = scaler.transform(Xtest)


CPU times: user 139 ms, sys: 55.8 ms, total: 195 ms
Wall time: 191 ms


In [16]:
np.savez("bdt_training_data", Xtrain_scaled=Xtrain_scaled, ytrain=ytrain, Xvalid_scaled=Xvalid_scaled, yvalid=yvalid, Xtest_scaled=Xtest_scaled, ytest=ytest, train_columns=train_columns)